# Esquema: clasificación multiclase (MVP manual)

Target **texto** → 0..K-1 manual. Misma idea: un pipeline por modelo, tabla de **accuracy**.

Siguiente: [07.b multiclase](../07.b-ejemplos-supervisados/03-clasificacion-multiple.ipynb).


## 1. CSV, tipos y faltantes

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("data/datos_flores.csv")
print(df.dtypes)
print("\nFaltantes:\n", df.isna().sum())


sepal_length    float64
sepal_width     float64
especie          object
dtype: object

Faltantes:
 sepal_length    1
sepal_width     1
especie         1
dtype: int64


## 2. Target multiclase → 0..K-1

In [7]:
df = df.dropna(subset=["especie"]).copy()
ORDEN_CLASES = ["setosa", "versicolor", "virginica"]
MAPA_MULTI = {n: i for i, n in enumerate(ORDEN_CLASES)}
y = df["especie"].str.strip().str.lower().map(MAPA_MULTI).astype(int)


## 3. Features numéricas

In [8]:
cols_num = ["sepal_length", "sepal_width"]
X = df[cols_num].astype(float).copy()
for col in cols_num:
    X[col] = X[col].fillna(X[col].median())


## 4. Split train / val / test (estratificado)


In [9]:
def split_train_val_test(X, y, test_size, val_size, random_state, stratify=False):
    """Divide en train, validación y test (dos llamadas a train_test_split).

    - test_size: fracción del total para test (hold-out final).
    - val_size: fracción de train+val → validación.
    Con test_size=0.2 y val_size=0.25 → ~60 % train, ~20 % val, ~20 % test.
    """
    kw = dict(test_size=test_size, random_state=random_state)
    if stratify:
        kw["stratify"] = y
    X_tv, X_test, y_tv, y_test = train_test_split(X, y, **kw)
    kw2 = dict(test_size=val_size, random_state=random_state)
    if stratify:
        kw2["stratify"] = y_tv
    X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, **kw2)
    return X_train, X_val, X_test, y_train, y_val, y_test


RANDOM_STATE = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.25

X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y, test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_STATE, stratify=True
)
print(f"Tamaños → train: {len(X_train)} | val: {len(X_val)} | test: {len(X_test)}")


Tamaños → train: 17 | val: 6 | test: 6


## 5. Entrenar modelos en Pipeline

Bucle sobre `build_models()`: **fit** en train; guardar pipelines en `pipelines` y predicciones en `predicciones_test`.

Solo **`fit` en train**; predicciones y métricas van en el apartado de análisis.


In [10]:
def build_models(n_classes):
    """Misma lista que 07.b (comenta entradas para excluir modelos)."""
    from sklearn.ensemble import (
        GradientBoostingClassifier,
        HistGradientBoostingClassifier,
        RandomForestClassifier,
    )
    from sklearn.linear_model import LogisticRegression, SGDClassifier
    from sklearn.multiclass import OneVsOneClassifier, OneVsRestClassifier
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.svm import SVC
    from sklearn.tree import DecisionTreeClassifier
    from xgboost import XGBClassifier
    from catboost import CatBoostClassifier

    if n_classes < 3:
        raise ValueError(f"build_models: K={n_classes} < 3 (multiclase)")
    return {
        "LogisticRegression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        "SGDClassifier": SGDClassifier(
            loss="log_loss",
            max_iter=2000,
            tol=1e-3,
            random_state=RANDOM_STATE,
        ),
        "SVC": SVC(random_state=RANDOM_STATE),
        "OneVsOneClassifier": OneVsOneClassifier(SVC(random_state=RANDOM_STATE)),
        "OneVsRestClassifier": OneVsRestClassifier(SVC(random_state=RANDOM_STATE)),
        "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        "DecisionTree": DecisionTreeClassifier(
            criterion="gini",
            splitter="best",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            min_weight_fraction_leaf=0.0,
            max_features=None,
            max_leaf_nodes=None,
            min_impurity_decrease=0.0,
            random_state=RANDOM_STATE,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=100,
            criterion="gini",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            min_weight_fraction_leaf=0.0,
            max_features="sqrt",
            max_leaf_nodes=None,
            min_impurity_decrease=0.0,
            bootstrap=True,
            oob_score=False,
            max_samples=None,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "HistGradientBoosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        "XGBoost": XGBClassifier(
            random_state=RANDOM_STATE,
            verbosity=0,
            n_estimators=100,
            objective="multi:softmax",
            num_class=n_classes,
            n_jobs=-1,
        ),
        "CatBoost": CatBoostClassifier(
            random_state=RANDOM_STATE,
            verbose=False,
            iterations=100,
            allow_writing_files=False,
            loss_function="MultiClass",
        ),
    }


RANDOM_STATE = 42
N_CLASSES = int(y.nunique())
MODELS = build_models(N_CLASSES)


pipelines = {}
for nombre, modelo in MODELS.items():
    pipe = make_pipeline(StandardScaler(), modelo)
    pipe.fit(X_train, y_train)
    pipelines[nombre] = pipe


## 6. Análisis comparativo

Tabla por **accuracy en val**; reporte en **test** del ganador.

Predicciones en **val** y **test**, tabla comparativa y elección del ganador por **val**.


In [11]:
filas = []
predicciones_test = {}
for nombre, pipe in pipelines.items():
    pred_val = pipe.predict(X_val)
    pred_test = pipe.predict(X_test)
    acc_val = accuracy_score(y_val, pred_val)
    acc_test = accuracy_score(y_test, pred_test)
    predicciones_test[nombre] = pred_test
    filas.append(
        {"modelo": nombre, "accuracy_val": acc_val, "accuracy_test": acc_test}
    )

tabla = pd.DataFrame(filas).sort_values("accuracy_val", ascending=False)
display(tabla.round(4))

mejor = tabla.iloc[0]
mejor_nombre = mejor["modelo"]
print(f"\nMejor accuracy en val: {mejor_nombre} ({mejor['accuracy_val']:.2f})")
print(f"Accuracy en test del ganador: {mejor['accuracy_test']:.2f}")
print(classification_report(
    y_test, predicciones_test[mejor_nombre], target_names=ORDEN_CLASES
))


,modelo,accuracy_val,accuracy_test
7,RandomForest,0.8333,0.3333
1,SGDClassifier,0.6667,0.6667
6,DecisionTree,0.6667,0.3333
10,XGBoost,0.6667,0.5000
11,CatBoost,0.6667,0.5000
8,GradientBoosting,0.6667,0.3333
0,LogisticRegression,0.5000,0.5000
2,SVC,0.5000,0.5000
3,OneVsOneClassifier,0.5000,0.5000
5,KNN,0.5000,0.3333



Mejor accuracy en val: RandomForest (0.83)
Accuracy en test del ganador: 0.33
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00         2
  versicolor       0.00      0.00      0.00         2
   virginica       0.00      0.00      0.00         2

    accuracy                           0.33         6
   macro avg       0.33      0.33      0.33         6
weighted avg       0.33      0.33      0.33         6

